In [ ]:
!pip install tensorflowjs

NameError: name 'tf' is not defined

In [ ]:
import tensorflowjs as tfjs
import tensorflow as tf
import os

# Define a dictionary for custom objects if your model uses any.
# Example: custom_objects = {'CustomLayerName': CustomLayerClass}
# If you don't have custom layers, you can leave this as an empty dictionary: custom_objects = {}
custom_objects = {}

# Load the original model
model_clean = tf.keras.models.load_model(
    '/content/drive/MyDrive/ClearQuote_Model/best_model_clean.keras',
    custom_objects=custom_objects
)

# --- Start of modification to remove problematic data augmentation layers ---
# Create a new model without the problematic data augmentation layers.
# Data augmentation layers are typically removed before deployment.

# Get the input layer of the original model
input_tensor = tf.keras.Input(shape=model_clean.input_shape[1:])
x = input_tensor

# Identify the starting index for layers to include in the new model.
# Based on the model summary, the first layer 'sequential' seems to be
# a data augmentation wrapper that was causing issues previously.
start_layer_to_include_index = 0
if len(model_clean.layers) > 0 and 'sequential' in model_clean.layers[0].name.lower():
    print(f"Skipping initial layer '{model_clean.layers[0].name}' (assumed to be data augmentation).")
    start_layer_to_include_index = 1
else:
    print("No initial 'sequential' layer found or model structure unexpected. Including all layers.")

# Iterate through the layers of the original model and rebuild from start_layer_to_include_index
for i in range(start_layer_to_include_index, len(model_clean.layers)):
    layer = model_clean.layers[i]
    try:
        x = layer(x)
    except Exception as e:
        print(f"Could not apply layer {layer.name} during model reconstruction. Error: {e}")
        print("This might happen if the model has a complex functional API structure.")
        print("Attempting to proceed, but manual adjustment might be needed.")
        raise # Re-raise if we can't handle it gracefully

# Create the new model without the skipped layers
new_model = tf.keras.Model(inputs=input_tensor, outputs=x)

print("Original model summary:")
model_clean.summary()
print("\nNew model summary (without problematic data augmentation layers):")
new_model.summary()

model_clean = new_model # Replace the original model with the new, cleaned one
# --- End of modification ---


# Define a temporary path for the SavedModel format
saved_model_path = '/tmp/saved_model_for_tfjs'

# Export the Keras model to TensorFlow's SavedModel format using model.export()
model_clean.export(saved_model_path)

# Convert the SavedModel to TensorFlow.js format
# Make sure the output directory for tfjs exists
output_dir = '/content/drive/MyDrive/tfjs_model'
os.makedirs(output_dir, exist_ok=True)
tfjs.converters.convert_tf_saved_model(saved_model_path, output_dir, control_flow_v2=True)

# Download the files from tfjs_model/ folder

Skipping initial layer 'sequential' (assumed to be data augmentation).
Original model summary:


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 300, 300, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb2 (Functional)     │ (None, 10, 10, 1408)   │     7,768,569 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1408)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1408)           │         5,632 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1408)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       360,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,566,200 (67.01 MB)

 Trainable params: 4,714,235 (17.98 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 300, 300, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb2 (Functional)     │ (None, 10, 10, 1408)   │     7,768,569 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1408)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1408)           │         5,632 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1408)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       360,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,137,728 (31.04 MB)

 Trainable params: 4,714,235 (17.98 MB)

 Non-trainable params: 3,423,493 (13.06 MB)

Saved artifact at '/tmp/saved_model_for_tfjs'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 300, 300, 3), dtype=tf.float32, name='keras_tensor_6226')
Output Type:
  TensorSpec(shape=(None, 7), dtype=tf.float32, name=None)
Captures:
  139158529410064: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  139158529416400: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  139158286063696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139158286062928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139158286061584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139158286064464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139158286065040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139158286065232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139158286063504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139158286064656: TensorSpec(shape=(), dtype=tf.resou

In [ ]:
import os
output_dir = '/content/drive/MyDrive/tfjs_model'
for f in os.listdir(output_dir):
    size = os.path.getsize(os.path.join(output_dir, f)) / (1024*1024)
    print(f"  {f:40s} {size:.2f} MB")

  group1-shard1of8.bin                     4.00 MB
  group1-shard2of8.bin                     4.00 MB
  group1-shard3of8.bin                     4.00 MB
  group1-shard4of8.bin                     4.00 MB
  group1-shard5of8.bin                     4.00 MB
  group1-shard6of8.bin                     4.00 MB
  group1-shard7of8.bin                     4.00 MB
  group1-shard8of8.bin                     2.70 MB
  model.json                               0.41 MB
